# SIR MODEL


# SIR WITH VACCINATION STATUS



The continuous SIR model is:

$$
\frac{dS}{dt} = -\beta SI -\delta I
$$

$$
\frac{dI}{dt} = \beta SI - \gamma I
$$

$$
\frac{dR}{dt} = \gamma I
$$

$$
\frac{dV}{dt} = \delta I
$$

where:

$$
S(t) = \text{susceptible population}
$$

$$
I(t) = \text{infectious population}
$$

$$
R(t) = \text{recovered population}
$$

$$
V(t) = \text{immunized population}
$$

$$
\beta = \text{transmission rate}
$$

$$
\gamma = \text{recovery rate}
$$

$$
\delta = \text{immunization rate}
$$
---




In [1]:
import numpy as np
import matplotlib.pyplot as plt
import ipywidgets as widgets
from scipy.integrate import solve_ivp
from IPython.display import display, clear_output

# -----------------------------
# Initial model parameters
# -----------------------------
initial_transmission_rate = 0.0035
initial_recovery_days = 7 # 7 days for recovery
initial_immunization_speed = 1 # Constant vaccination rate of 1 person per day


S0 = 249
I0 = 1
R0 = 0
V0 = 0

days = 160
dt = 0.1
time = np.arange(0, days, dt)


# -----------------------------
# SIR simulation function
# -----------------------------
def simulate_sir(transmission_rate, recovery_days, vaccination_speed):
    recovery_rate = 1 / recovery_days

    def sir_system(t, state):
        S, I, R, V = state

        # Avoid numerical negatives inside the ODE logic
        S = max(S, 0)
        I = max(I, 0)

        new_infections = transmission_rate * S * I
        new_recoveries = recovery_rate * I

        # Constant vaccination speed: max 1 person/day,
        # but never vaccinate more susceptible people than remain.
        new_vaccinations = min(vaccination_speed, S)

        dS_dt = -new_infections - new_vaccinations
        dI_dt = new_infections - new_recoveries
        dR_dt = new_recoveries
        dV_dt = new_vaccinations

        return [dS_dt, dI_dt, dR_dt, dV_dt]

    solution = solve_ivp(
        sir_system,
        (time[0], time[-1]),
        [S0, I0, R0, V0],
        t_eval=time,
    )

    if not solution.success:
        raise RuntimeError(f"SIR integration failed: {solution.message}")

    # Remove tiny numerical negative values caused by the solver
    return np.maximum(solution.y, 0)


# -----------------------------
# Widgets
# -----------------------------
beta_slider = widgets.FloatSlider(
    value=initial_transmission_rate,
    min=0.0001,
    max=0.01,
    step=0.0001,
    description="β:",
    readout_format=".4f"
)

recovery_slider = widgets.IntSlider(
    value=initial_recovery_days,
    min=1,
    max=30,
    step=1,
    description="Recovery days:"
)

reset_button = widgets.Button(
    description="Reset",
    button_style="warning"
)

output = widgets.Output()


# -----------------------------
# Plot function
# -----------------------------
def draw_plot():
    transmission_rate = beta_slider.value
    recovery_days = recovery_slider.value
    vaccination_speed = initial_immunization_speed

    S, I, R, V = simulate_sir(transmission_rate, recovery_days, vaccination_speed)

    with output:
        clear_output(wait=True)

        fig, ax = plt.subplots(figsize=(10, 6))

        ax.plot(time, S, label="Susceptible S(t)")
        ax.plot(time, I, label="Infectious I(t)")
        ax.plot(time, R, label="Recovered R(t)")
        ax.plot(time, V, label="Vaccinated V(t)")

        ax.set_title(
            f"SIR Model: transmission rate β = {transmission_rate:.4f}, "
            f"recovery time = {recovery_days} days, "
            f"vaccination speed = {vaccination_speed} per day"
        )
        ax.set_xlabel("Time in days")
        ax.set_ylabel("Number of people")
        ax.set_ylim(0, max(S.max(), I.max(), R.max(), V.max()) * 1.1)
        ax.legend()
        ax.grid(True)

        plt.show()
        plt.close(fig)


# -----------------------------
# Update function
# -----------------------------
def update_plot(change=None):
    draw_plot()


# -----------------------------
# Reset function
# -----------------------------
def reset_model(button):
    beta_slider.unobserve(update_plot, names="value")
    recovery_slider.unobserve(update_plot, names="value")

    beta_slider.value = initial_transmission_rate
    recovery_slider.value = initial_recovery_days

    beta_slider.observe(update_plot, names="value")
    recovery_slider.observe(update_plot, names="value")

    draw_plot()


beta_slider.observe(update_plot, names="value")
recovery_slider.observe(update_plot, names="value")
reset_button.on_click(reset_model)


# Clear previous notebook output before displaying again
clear_output(wait=True)

display(beta_slider, recovery_slider, reset_button, output)

draw_plot()

FloatSlider(value=0.0035, description='β:', max=0.01, min=0.0001, readout_format='.4f', step=0.0001)

IntSlider(value=7, description='Recovery days:', max=30, min=1)

Button(button_style='warning', description='Reset', style=ButtonStyle())

Output()